<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 02 — Entendendo os dados com o Assistant (RavenStack)

1. Rode o **LAB 01** antes (tabelas Delta no mesmo `catalog_name` / `schema_name`).
2. Confira os widgets.
3. Cluster ou serverless → **Run All**.
4. Use o Assistant (`Ctrl + I`) só com as colunas listadas no Lab 01.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "ravenstack")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

### Passo 1 — Perguntas abertas (Databricks Assistant)

1. Copie uma pergunta.
2. Clique no Assistant ao lado da célula SQL ou use `Ctrl + I`.
3. Peça SQL que use **somente** as colunas das tabelas `ravenstack_*`.
4. Execute o código sugerido.

Não pergunte sobre faturas, CNPJ, bairros ou inadimplência — essas colunas **não existem** nestes CSVs.

## Perguntas (colunas reais do CSV)

- Quantas contas distintas existem em `ravenstack_accounts` e quantas têm pelo menos um `churn_event_id`?
- O `churn_flag` de `ravenstack_accounts` coincide com a existência de `churn_event_id`? Liste as divergências (`account_id`).
- Taxa de contas com evento de churn por `industry`, `country`, `referral_source` e `plan_tier`.
- `arr_amount` da assinatura mais recente sem `end_date` nas contas que já têm `churn_event`.
- 10 `account_id` com maior `arr_amount` (assinatura vigente) e pelo menos um `churn_event`.
- `reason_code` mais frequente e `refund_amount_usd` médio por reason.
- `resolution_time_hours` e `satisfaction_score` médios: contas com churn event vs sem.
- `usage_count` e `error_count` médios por `subscription_id` (join `subscriptions.account_id`) em contas com vs sem evento.
- Contas com `downgrade_flag` verdadeiro em alguma assinatura e depois um `churn_event`.

In [ ]:
display(
    spark.sql(
        """
SELECT
  (SELECT COUNT(*) FROM ravenstack_accounts) AS accounts,
  (SELECT COUNT(*) FROM ravenstack_subscriptions) AS subscriptions,
  (SELECT COUNT(*) FROM ravenstack_feature_usage) AS feature_usage,
  (SELECT COUNT(*) FROM ravenstack_support_tickets) AS support_tickets,
  (SELECT COUNT(*) FROM ravenstack_churn_events) AS churn_events,
  (SELECT COUNT(DISTINCT account_id) FROM ravenstack_churn_events) AS accounts_with_churn_event
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  a.churn_flag AS account_churn_flag,
  CASE WHEN e.account_id IS NOT NULL THEN true ELSE false END AS has_churn_event,
  COUNT(*) AS accounts
FROM ravenstack_accounts a
LEFT JOIN (
  SELECT DISTINCT account_id FROM ravenstack_churn_events
) e ON a.account_id = e.account_id
GROUP BY a.churn_flag, CASE WHEN e.account_id IS NOT NULL THEN true ELSE false END
ORDER BY 1, 2
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT a.industry,
       COUNT(*) AS accounts,
       SUM(CASE WHEN e.account_id IS NOT NULL THEN 1 ELSE 0 END) AS with_churn_event,
       ROUND(AVG(CASE WHEN e.account_id IS NOT NULL THEN 1.0 ELSE 0.0 END), 3) AS churn_event_rate
FROM ravenstack_accounts a
LEFT JOIN (
  SELECT DISTINCT account_id FROM ravenstack_churn_events
) e ON a.account_id = e.account_id
GROUP BY a.industry
ORDER BY churn_event_rate DESC
"""
    )
)

### Bônus — prompt para o Assistant

> Estude as colunas de `ravenstack_accounts`, `ravenstack_subscriptions`, `ravenstack_feature_usage`, `ravenstack_support_tickets` e `ravenstack_churn_events`. Cruze por `account_id` e `subscription_id`. Use `churn_events` como label de churn (não `accounts.churn_flag`). Liste segmentos em risco com `account_id` e estime receita (`arr_amount` da assinatura vigente). Não invente colunas.

### Genie

Sala publicada: [lab-001-churn-genie](https://dbc-0dea39b3-be1e.cloud.databricks.com/genie/rooms/01f1b6165ad01ca4a5f3b95c6d3d3609?o=7474644277755368) (`01f1b6165ad01ca4a5f3b95c6d3d3609`).

O chatbot local (`e2e-chatbot-app-next`) abre esta sala em `/room/churn`.

Label = ravenstack_churn_events. Nao usar accounts.churn_flag como verdade.
